In [10]:


from scapy.all import rdpcap, IP, TCP, UDP
import pandas as pd
from collections import defaultdict


# Input file (Change this to your pcap file)
PCAP_FILE = "dump.pcap"

# Data storage for flows
flows = defaultdict(lambda: {
    "IPV4_SRC_ADDR": None, "IPV4_DST_ADDR": None,
    "L4_SRC_PORT": None, "L4_DST_PORT": None,
    "PROTOCOL": None, "TCP_FLAGS": 0, "L7_PROTO": None,
    "IN_BYTES": 0, "OUT_BYTES": 0,
    "IN_PKTS": 0, "OUT_PKTS": 0, "FLOW_DURATION_MILLISECONDS": 0
})

# Read packets from pcap file
packets = rdpcap(PCAP_FILE)

# Track timestamps
flow_start_times = {}

for pkt in packets:
    if IP in pkt:
        src_ip = pkt[IP].src
        dst_ip = pkt[IP].dst
        proto = pkt[IP].proto
        timestamp = pkt.time  # Packet timestamp

        # Identify Layer 4 details
        if TCP in pkt or UDP in pkt:
            src_port = pkt[TCP].sport if TCP in pkt else pkt[UDP].sport
            dst_port = pkt[TCP].dport if TCP in pkt else pkt[UDP].dport
        else:
            src_port, dst_port = None, None
        
        # Create flow key (bidirectional)
        flow_key = (src_ip, dst_ip, src_port, dst_port, proto)
        reverse_key = (dst_ip, src_ip, dst_port, src_port, proto)

        # Determine direction (incoming/outgoing)
        if flow_key in flows:
            direction = "OUT"
        elif reverse_key in flows:
            flow_key = reverse_key
            direction = "IN"
        else:
            direction = "OUT"  # Default assumption

        flow = flows[flow_key]
        
        # Set IP & Port details
        flow["IPV4_SRC_ADDR"] = flow_key[0]
        flow["IPV4_DST_ADDR"] = flow_key[1]
        flow["L4_SRC_PORT"] = flow_key[2]
        flow["L4_DST_PORT"] = flow_key[3]
        flow["PROTOCOL"] = proto

        # TCP flag aggregation
        if TCP in pkt:
            flow["TCP_FLAGS"] |= pkt[TCP].flags

        # Packet size updates
        pkt_size = len(pkt)
        if direction == "OUT":
            flow["OUT_BYTES"] += pkt_size
            flow["OUT_PKTS"] += 1
        else:
            flow["IN_BYTES"] += pkt_size
            flow["IN_PKTS"] += 1

        # Track flow start & duration
        if flow_key not in flow_start_times:
            flow_start_times[flow_key] = timestamp
        flow["FLOW_DURATION_MILLISECONDS"] = int((timestamp - flow_start_times[flow_key]) * 1000)

# Convert flow data to pandas DataFrame
df = pd.DataFrame(flows.values())

# Save to CSV
df.to_csv("network_flow_dataset.csv", index=False)

print("Network flow dataset saved as 'network_flow_dataset.csv'")


Network flow dataset saved as 'network_flow_dataset.csv'
